# Stage 6: Multi-Color Classification

This notebook extends our color extraction to classify garments as **solid** or **multi-color**.

**Objectives:**
- Add solid vs multi-color classification based on dominant color percentage
- Define color pattern categories (solid, two-tone, multi-color)
- Update harmony analysis to consider multiple colors
- Test on sample images

In [ ]:
# Setup and imports
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional
from enum import Enum

from src.attributes.color_extractor import ColorExtractor, DominantColor, ColorHarmonyAnalyzer

print("Imports successful!")

## 6.1 Color Pattern Classification

We classify garments based on the distribution of their dominant colors:

| Pattern Type | Criteria |
|--------------|----------|
| **Solid** | Primary color >= 85% |
| **Two-tone** | Primary color 50-85%, secondary >= 15% |
| **Multi-color** | Primary color < 50% OR 3+ significant colors |

In [ ]:
class ColorPattern(Enum):
    """Classification of garment color patterns."""
    SOLID = "solid"
    TWO_TONE = "two-tone"
    MULTI_COLOR = "multi-color"


@dataclass
class ColorClassification:
    """Result of color pattern classification."""
    pattern: ColorPattern
    primary_color: DominantColor
    secondary_color: Optional[DominantColor]
    accent_colors: List[DominantColor]
    confidence: float
    
    def __str__(self):
        colors_str = self.primary_color.name
        if self.secondary_color:
            colors_str += f" + {self.secondary_color.name}"
        if self.accent_colors:
            colors_str += f" + {len(self.accent_colors)} accent(s)"
        return f"{self.pattern.value}: {colors_str} ({self.confidence:.0%})"


print("ColorPattern enum and ColorClassification defined!")

In [ ]:
class MultiColorClassifier:
    """
    Classify garments as solid, two-tone, or multi-color.
    
    Uses the distribution of dominant colors to determine the pattern type.
    """
    
    def __init__(
        self,
        solid_threshold: float = 0.85,
        two_tone_threshold: float = 0.50,
        min_secondary_pct: float = 0.15,
        min_accent_pct: float = 0.05
    ):
        """
        Initialize the classifier.
        
        Args:
            solid_threshold: Min percentage for primary color to be 'solid'
            two_tone_threshold: Min percentage for primary color in 'two-tone'
            min_secondary_pct: Min percentage for a secondary color
            min_accent_pct: Min percentage for an accent color
        """
        self.solid_threshold = solid_threshold
        self.two_tone_threshold = two_tone_threshold
        self.min_secondary_pct = min_secondary_pct
        self.min_accent_pct = min_accent_pct
    
    def classify(self, colors: List[DominantColor]) -> ColorClassification:
        """
        Classify the color pattern from extracted dominant colors.
        
        Args:
            colors: List of DominantColor objects (sorted by percentage, descending)
        
        Returns:
            ColorClassification with pattern type and color breakdown
        """
        if not colors:
            raise ValueError("No colors provided for classification")
        
        primary = colors[0]
        secondary = colors[1] if len(colors) > 1 else None
        
        # Count significant colors
        significant_colors = [c for c in colors if c.percentage >= self.min_accent_pct]
        
        # Determine pattern type
        if primary.percentage >= self.solid_threshold:
            # Solid: one dominant color
            pattern = ColorPattern.SOLID
            confidence = primary.percentage
            secondary_out = None
            accents = []
            
        elif primary.percentage >= self.two_tone_threshold:
            # Check if there's a significant secondary color
            if secondary and secondary.percentage >= self.min_secondary_pct:
                pattern = ColorPattern.TWO_TONE
                confidence = primary.percentage + secondary.percentage
                secondary_out = secondary
                accents = [c for c in colors[2:] if c.percentage >= self.min_accent_pct]
            else:
                # Primary is dominant but not quite solid
                pattern = ColorPattern.SOLID
                confidence = primary.percentage
                secondary_out = None
                accents = []
                
        else:
            # Multi-color: many significant colors or no single dominant
            pattern = ColorPattern.MULTI_COLOR
            confidence = sum(c.percentage for c in significant_colors)
            secondary_out = secondary if secondary and secondary.percentage >= self.min_secondary_pct else None
            accents = [c for c in colors[2:] if c.percentage >= self.min_accent_pct]
        
        return ColorClassification(
            pattern=pattern,
            primary_color=primary,
            secondary_color=secondary_out,
            accent_colors=accents,
            confidence=confidence
        )


# Initialize classifier
mc_classifier = MultiColorClassifier()
print("MultiColorClassifier initialized!")
print(f"  Solid threshold: {mc_classifier.solid_threshold:.0%}")
print(f"  Two-tone threshold: {mc_classifier.two_tone_threshold:.0%}")

## 6.2 Test with Synthetic Images

Let's create test images representing different color patterns.

In [ ]:
def create_solid_image(color_rgb, size=224):
    """Create a solid color image."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[:, :] = color_rgb
    return img

def create_two_tone_image(color1_rgb, color2_rgb, split=0.5, size=224):
    """Create a two-tone image (horizontal split)."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    split_row = int(size * split)
    img[:split_row, :] = color1_rgb
    img[split_row:, :] = color2_rgb
    return img

def create_striped_image(colors_rgb, stripe_width=20, size=224):
    """Create a striped image with multiple colors."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    n_colors = len(colors_rgb)
    for i in range(0, size, stripe_width):
        color_idx = (i // stripe_width) % n_colors
        img[i:i+stripe_width, :] = colors_rgb[color_idx]
    return img

def create_multicolor_blocks(colors_rgb, size=224):
    """Create an image with color blocks."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    n_colors = len(colors_rgb)
    block_size = size // int(np.ceil(np.sqrt(n_colors)))
    
    idx = 0
    for i in range(0, size, block_size):
        for j in range(0, size, block_size):
            if idx < n_colors:
                img[i:i+block_size, j:j+block_size] = colors_rgb[idx]
                idx += 1
    return img

print("Image creation functions defined!")

In [ ]:
# Create test images
test_images = {
    "Solid Navy": create_solid_image([30, 50, 100]),
    "Solid Red": create_solid_image([200, 50, 50]),
    "Two-Tone (Navy/White)": create_two_tone_image([30, 50, 100], [255, 255, 255]),
    "Two-Tone (Black/Gray)": create_two_tone_image([20, 20, 20], [150, 150, 150]),
    "Striped (3 colors)": create_striped_image([[200, 50, 50], [255, 255, 255], [30, 50, 100]]),
    "Multi-Color Blocks": create_multicolor_blocks([[200, 50, 50], [50, 150, 50], [50, 50, 200], [255, 200, 50]])
}

# Visualize test images
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, img) in zip(axes.flat, test_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6.3 Run Classification on Test Images

In [ ]:
# Initialize color extractor
color_extractor = ColorExtractor(
    n_colors=5,
    remove_background=False,  # No background in synthetic images
    filter_skin_tones=False
)

# Classify each test image
results = {}
for name, img in test_images.items():
    colors = color_extractor.extract(img)
    classification = mc_classifier.classify(colors)
    results[name] = (colors, classification)
    
    print(f"\n{name}:")
    print(f"  Classification: {classification}")
    print(f"  Colors found: {len(colors)}")
    for i, c in enumerate(colors):
        print(f"    {i+1}. {c.name}: {c.percentage:.1%}")

In [ ]:
def visualize_classification(img, colors, classification, title=""):
    """Visualize image with its color classification."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title(title or "Image")
    axes[0].axis('off')
    
    # Color breakdown
    left = 0
    for c in colors:
        axes[1].barh(0, c.percentage, left=left, 
                     color=np.array(c.rgb)/255, edgecolor='black', height=0.8)
        if c.percentage > 0.1:
            axes[1].text(left + c.percentage/2, 0, f"{c.name}\n{c.percentage:.0%}",
                        ha='center', va='center', fontsize=8)
        left += c.percentage
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(-0.5, 0.5)
    axes[1].set_title("Color Breakdown")
    axes[1].axis('off')
    
    # Classification result
    axes[2].text(0.5, 0.6, classification.pattern.value.upper(), 
                fontsize=24, ha='center', va='center', fontweight='bold',
                color='darkblue')
    axes[2].text(0.5, 0.35, f"Confidence: {classification.confidence:.0%}",
                fontsize=12, ha='center', va='center')
    
    # Color dots
    color_y = 0.15
    if classification.primary_color:
        axes[2].scatter([0.3], [color_y], c=[np.array(classification.primary_color.rgb)/255], 
                       s=200, edgecolors='black')
        axes[2].text(0.3, 0.02, "Primary", ha='center', fontsize=8)
    if classification.secondary_color:
        axes[2].scatter([0.5], [color_y], c=[np.array(classification.secondary_color.rgb)/255],
                       s=200, edgecolors='black')
        axes[2].text(0.5, 0.02, "Secondary", ha='center', fontsize=8)
    if classification.accent_colors:
        for i, ac in enumerate(classification.accent_colors[:2]):
            axes[2].scatter([0.7 + i*0.15], [color_y], c=[np.array(ac.rgb)/255],
                           s=100, edgecolors='black')
        axes[2].text(0.75, 0.02, "Accents", ha='center', fontsize=8)
    
    axes[2].set_xlim(0, 1)
    axes[2].set_ylim(0, 0.8)
    axes[2].set_title("Classification")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()


# Visualize all results
for name, (colors, classification) in results.items():
    visualize_classification(test_images[name], colors, classification, name)

## 6.4 Enhanced Color Extractor

Now let's create an enhanced version of the ColorExtractor that includes multi-color classification.

In [ ]:
class EnhancedColorExtractor(ColorExtractor):
    """
    Extended ColorExtractor with multi-color classification support.
    
    Adds the ability to classify garments as solid, two-tone, or multi-color
    based on the distribution of dominant colors.
    """
    
    def __init__(
        self,
        n_colors: int = 5,
        color_space: str = "LAB",
        min_percentage: float = 0.05,
        remove_background: bool = True,
        filter_skin_tones: bool = True,
        solid_threshold: float = 0.85,
        two_tone_threshold: float = 0.50
    ):
        super().__init__(
            n_colors=n_colors,
            color_space=color_space,
            min_percentage=min_percentage,
            remove_background=remove_background,
            filter_skin_tones=filter_skin_tones
        )
        
        self.mc_classifier = MultiColorClassifier(
            solid_threshold=solid_threshold,
            two_tone_threshold=two_tone_threshold,
            min_secondary_pct=min_percentage,
            min_accent_pct=min_percentage
        )
    
    def extract_with_classification(self, image) -> Tuple[List[DominantColor], ColorClassification]:
        """
        Extract colors and classify the color pattern.
        
        Args:
            image: Input image
            
        Returns:
            Tuple of (dominant_colors, classification)
        """
        colors = self.extract(image)
        classification = self.mc_classifier.classify(colors)
        return colors, classification
    
    def is_solid(self, image) -> bool:
        """Quick check if an image is a solid color."""
        _, classification = self.extract_with_classification(image)
        return classification.pattern == ColorPattern.SOLID
    
    def is_multicolor(self, image) -> bool:
        """Quick check if an image is multi-colored."""
        _, classification = self.extract_with_classification(image)
        return classification.pattern == ColorPattern.MULTI_COLOR


# Test the enhanced extractor
enhanced_extractor = EnhancedColorExtractor(
    remove_background=False,
    filter_skin_tones=False
)

print("EnhancedColorExtractor initialized!")

In [ ]:
# Test enhanced extractor on all images
print("Testing EnhancedColorExtractor:\n")

for name, img in test_images.items():
    colors, classification = enhanced_extractor.extract_with_classification(img)
    print(f"{name}:")
    print(f"  is_solid: {enhanced_extractor.is_solid(img)}")
    print(f"  is_multicolor: {enhanced_extractor.is_multicolor(img)}")
    print(f"  pattern: {classification.pattern.value}")
    print()

## 6.5 Updated Harmony Analysis for Multi-Color Items

The original harmony analyzer only considers the primary color. Let's create an enhanced version that considers secondary colors too.

In [ ]:
class EnhancedHarmonyAnalyzer(ColorHarmonyAnalyzer):
    """
    Enhanced color harmony analyzer that considers multiple colors.
    
    Extends the base analyzer to handle:
    - Solid + Solid pairing
    - Solid + Multi-color pairing
    - Multi-color + Multi-color pairing
    """
    
    def analyze_harmony_enhanced(
        self,
        classification1: ColorClassification,
        classification2: ColorClassification
    ) -> dict:
        """
        Analyze harmony considering the full color classification.
        
        Args:
            classification1: ColorClassification for first item
            classification2: ColorClassification for second item
            
        Returns:
            Dictionary with harmony analysis
        """
        p1 = classification1.pattern
        p2 = classification2.pattern
        
        # Get all significant colors for each item
        colors1 = [classification1.primary_color]
        if classification1.secondary_color:
            colors1.append(classification1.secondary_color)
        colors1.extend(classification1.accent_colors)
        
        colors2 = [classification2.primary_color]
        if classification2.secondary_color:
            colors2.append(classification2.secondary_color)
        colors2.extend(classification2.accent_colors)
        
        # Base harmony from primary colors
        base_harmony = self.analyze_harmony(
            [classification1.primary_color],
            [classification2.primary_color]
        )
        
        # Adjust based on pattern combination
        score = base_harmony['score']
        pattern_note = ""
        
        # Solid + Solid: Use base harmony
        if p1 == ColorPattern.SOLID and p2 == ColorPattern.SOLID:
            pattern_note = "Both solid - classic pairing"
            
        # Solid + Multi-color: Check if solid color appears in multi-color
        elif p1 == ColorPattern.SOLID and p2 in (ColorPattern.TWO_TONE, ColorPattern.MULTI_COLOR):
            # Bonus if solid matches one of the colors in multi-color item
            if self._color_matches_any(classification1.primary_color, colors2):
                score += 0.1
                pattern_note = "Solid matches multi-color element - cohesive"
            else:
                pattern_note = "Solid with patterned - check contrast"
                
        elif p2 == ColorPattern.SOLID and p1 in (ColorPattern.TWO_TONE, ColorPattern.MULTI_COLOR):
            if self._color_matches_any(classification2.primary_color, colors1):
                score += 0.1
                pattern_note = "Solid matches multi-color element - cohesive"
            else:
                pattern_note = "Solid with patterned - check contrast"
                
        # Multi-color + Multi-color: Generally harder to match
        elif p1 in (ColorPattern.TWO_TONE, ColorPattern.MULTI_COLOR) and \
             p2 in (ColorPattern.TWO_TONE, ColorPattern.MULTI_COLOR):
            # Check for shared colors
            shared = self._count_shared_colors(colors1, colors2)
            if shared >= 1:
                score += 0.05 * shared
                pattern_note = f"Patterns share {shared} color(s) - coordinated"
            else:
                score -= 0.1
                pattern_note = "Multiple patterns without shared colors - busy"
        
        score = min(1.0, max(0.0, score))
        
        return {
            'harmony_type': base_harmony['harmony_type'],
            'score': score,
            'pattern_combination': f"{p1.value} + {p2.value}",
            'pattern_note': pattern_note,
            'colors_item1': [c.name for c in colors1],
            'colors_item2': [c.name for c in colors2]
        }
    
    def _color_matches_any(self, color: DominantColor, color_list: List[DominantColor], 
                           threshold: float = 30.0) -> bool:
        """Check if a color closely matches any color in a list (using LAB distance)."""
        from scipy.spatial import distance
        color_lab = np.array(color.lab)
        for c in color_list:
            if distance.euclidean(color_lab, np.array(c.lab)) < threshold:
                return True
        return False
    
    def _count_shared_colors(self, colors1: List[DominantColor], 
                             colors2: List[DominantColor], threshold: float = 30.0) -> int:
        """Count how many colors are shared between two lists."""
        shared = 0
        for c1 in colors1:
            if self._color_matches_any(c1, colors2, threshold):
                shared += 1
        return shared


# Initialize enhanced analyzer
enhanced_harmony = EnhancedHarmonyAnalyzer()
print("EnhancedHarmonyAnalyzer initialized!")

In [ ]:
# Test enhanced harmony analysis
print("Testing Enhanced Harmony Analysis:\n")

# Get classifications for test images
classifications = {}
for name, img in test_images.items():
    _, classification = enhanced_extractor.extract_with_classification(img)
    classifications[name] = classification

# Test pairings
test_pairs = [
    ("Solid Navy", "Solid Red"),
    ("Solid Navy", "Two-Tone (Navy/White)"),
    ("Solid Red", "Striped (3 colors)"),
    ("Two-Tone (Navy/White)", "Striped (3 colors)"),
    ("Striped (3 colors)", "Multi-Color Blocks"),
]

for name1, name2 in test_pairs:
    c1 = classifications[name1]
    c2 = classifications[name2]
    
    result = enhanced_harmony.analyze_harmony_enhanced(c1, c2)
    
    print(f"{name1} + {name2}:")
    print(f"  Pattern combo: {result['pattern_combination']}")
    print(f"  Harmony type: {result['harmony_type']}")
    print(f"  Score: {result['score']:.2f}")
    print(f"  Note: {result['pattern_note']}")
    print()

## 6.6 Test on Real Images (Optional)

If you have sample images, test the multi-color classification on them.

In [ ]:
# Test on real images if available
sample_dir = Path('../data/samples')

# Re-initialize with background removal for real images
real_extractor = EnhancedColorExtractor(
    remove_background=True,
    filter_skin_tones=True
)

if sample_dir.exists():
    for img_path in list(sample_dir.glob('*.jpg'))[:4]:  # First 4 images
        img = np.array(Image.open(img_path))
        colors, classification = real_extractor.extract_with_classification(img)
        
        visualize_classification(img, colors, classification, img_path.name)
else:
    print(f"No sample images found in {sample_dir}")
    print("Add .jpg images to test on real garments.")

## 6.7 Save the Module

Let's save the new classes to a module file for use in other notebooks.

In [ ]:
# The classes we created in this notebook have been saved to:
# src/attributes/multicolor.py
#
# You can import them like:
# from src.attributes.multicolor import (
#     ColorPattern,
#     ColorClassification,
#     MultiColorClassifier,
#     EnhancedColorExtractor,
#     EnhancedHarmonyAnalyzer
# )

print("Module saved to src/attributes/multicolor.py")
print("\nAvailable classes:")
print("  - ColorPattern: Enum (SOLID, TWO_TONE, MULTI_COLOR)")
print("  - ColorClassification: Dataclass for classification results")
print("  - MultiColorClassifier: Classify colors into patterns")
print("  - EnhancedColorExtractor: ColorExtractor with classification")
print("  - EnhancedHarmonyAnalyzer: Harmony analysis for multi-color items")

## Summary

In this notebook we:

1. **Created color pattern classification** - Categorizes garments as solid, two-tone, or multi-color based on dominant color percentages

2. **Built the MultiColorClassifier** - Uses configurable thresholds:
   - Solid: Primary color >= 85%
   - Two-tone: Primary 50-85% with significant secondary
   - Multi-color: Primary < 50% or 3+ colors

3. **Created EnhancedColorExtractor** - Extends the original with:
   - `extract_with_classification()` - Returns colors and classification
   - `is_solid()` / `is_multicolor()` - Quick pattern checks

4. **Built EnhancedHarmonyAnalyzer** - Better harmony scoring for:
   - Solid + patterned combinations
   - Detecting shared colors between multi-color items
   - Penalizing clashing patterns

**Next Steps:**
- Integrate into the main pipeline
- Test with more real images
- Consider training a pattern detector (stripes, plaid, floral) for more granular classification